In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.stats import mannwhitneyu

In [22]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

def test_plot_m1_frac(tumor_type):
    # Load xCell & CNV data
    xcell_path = f"/Users/annabelshinichen/Desktop/SchoolWork/MSBMI/Davoli_lab/TCGA/xcell_TCGA/{tumor_type}_xcell.tsv"
    cnv_path = f"/Users/annabelshinichen/Desktop/SchoolWork/MSBMI/Davoli_lab/TCGA/cnvtable_purity/cnv_table_all/{tumor_type}_2020-06-11_purity_rescale_cnv_table.txt"
    
    xcell = pd.read_csv(xcell_path, sep="\t", index_col=0)
    cnv = pd.read_csv(cnv_path, sep="\t", index_col=0)
    cnv.columns = cnv.columns.str.replace("^X", "", regex=True)

    # Normalize sample IDs
    xcell.index = xcell.index.str.slice(0, 12)
    cnv.index = cnv.index.str.slice(0, 12)

    # Compute M1 / (M1 + M2)
    x = xcell[["Macrophage M1", "Macrophage M2"]].copy()
    x["M1_frac"] = x["Macrophage M1"] / (x["Macrophage M1"] + x["Macrophage M2"] + 1e-6)

    # Merge
    merged = x.merge(cnv, left_index=True, right_index=True)
    if "Arm1q" not in merged.columns:
        print(f"[ERROR] Arm1q not found in CNV data for {tumor_type}")
        return

    # Classify 1q status
    merged["Arm_1q_status"] = merged["Arm1q"].apply(
        lambda v: "Gain" if v > 0.2 else "Neutral" if v > -0.2 else "Loss"
    )
    filtered = merged[merged["Arm_1q_status"].isin(["Gain", "Neutral"])]

    # Wilcoxon test
    gain = filtered[filtered["Arm_1q_status"] == "Gain"]["M1_frac"]
    neutral = filtered[filtered["Arm_1q_status"] == "Neutral"]["M1_frac"]
    stat, pval = mannwhitneyu(gain, neutral, alternative="two-sided")

    # Plot
    plt.figure(figsize=(6, 5))
    sns.boxplot(
        data=filtered,
        x="Arm_1q_status",
        y="M1_frac",
        palette={"Gain": "red", "Neutral": "gray"}
    )
    sns.stripplot(
        data=filtered,
        x="Arm_1q_status",
        y="M1_frac",
        color="black",
        size=3,
        jitter=True,
        alpha=0.4
    )
    plt.title(f"{tumor_type}: M1 / (M1 + M2) vs 1q CNV Status")
    plt.ylabel("M1 / (M1 + M2)")
    plt.xlabel("1q CNV Status")

    # Annotate p-value
    ymin = filtered["M1_frac"].min()
    plt.text(0.5, ymin - 0.05 * abs(ymin), f"p = {pval:.4g}", ha="center", fontsize=12)

    plt.tight_layout()
    plt.savefig(f"{tumor_type}_M1_fraction_boxplot.pdf")
    plt.close()

    # Console output
    print(f"Done: {tumor_type} | Wilcoxon p = {pval:.4g}")

In [23]:
test_plot_m1_frac("LUAD")
test_plot_m1_frac("HNSC")
test_plot_m1_frac("UCEC")
test_plot_m1_frac("PAAD")
test_plot_m1_frac("SKCM")

/var/folders/6l/7vr5y5g91fb54d47lr02cq2h0000gn/T/ipykernel_49460/3247231531.py:43: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(


Done: LUAD | Wilcoxon p = 0.5394
Done: HNSC | Wilcoxon p = 0.06088


/var/folders/6l/7vr5y5g91fb54d47lr02cq2h0000gn/T/ipykernel_49460/3247231531.py:43: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
/var/folders/6l/7vr5y5g91fb54d47lr02cq2h0000gn/T/ipykernel_49460/3247231531.py:43: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(


Done: UCEC | Wilcoxon p = 0.2917
Done: PAAD | Wilcoxon p = 0.4956


/var/folders/6l/7vr5y5g91fb54d47lr02cq2h0000gn/T/ipykernel_49460/3247231531.py:43: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
/var/folders/6l/7vr5y5g91fb54d47lr02cq2h0000gn/T/ipykernel_49460/3247231531.py:43: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(


Done: SKCM | Wilcoxon p = 0.1108
